In [1]:
import matplotlib.pyplot as plt
import struct
import pandas as pd

def parse_qbb_trace(file_path):
    """
    Parses the binary QBB trace file (TraceFormat struct, 56 bytes each).

    This version matches the exact C++ struct definition and field layout.
    It decodes the union contents based on l3Prot:
      0x6   = TCP   -> data
      0x11  = UDP   -> data
      0xFC/0xFD = ACK
      0xFE  = PFC
      0xFF  = CNP
      else  = qp (default)
    """

    # --- Base struct before the union (12 fields) ---
    base_fmt = "<QHBBIIIHBBBB"
    base_size = struct.calcsize(base_fmt)

    # --- Union structs (from TraceFormat union) ---
    fmt_data = "<HHIQHH"      # sport, dport, seq, ts, pg, payload
    fmt_ack  = "<HHHHIQ"      # sport, dport, flags, pg, seq, ts
    fmt_pfc  = "<IIb3x"       # time, qlen, qIndex (+3 padding to 12 bytes)
    fmt_cnp  = "<HBBHH"       # fid, qIndex, ecnBits, qfb, total
    fmt_qp   = "<HH"          # sport, dport

    rec_size = 56  # confirmed from debugger
    rows = []

    # --- Read file into memory ---
    try:
        with open(file_path, "rb") as f:
            data = f.read()
    except FileNotFoundError:
        print(f"Error: file not found: {file_path}")
        return pd.DataFrame()

    n = len(data)
    if n < rec_size:
        print("Trace file too small — no complete records.")
        return pd.DataFrame()

    off = 0
    last_time = -1

    while off + rec_size <= n:
        # Unpack header (base)
        try:
            header = struct.unpack_from(base_fmt, data, off)
        except struct.error:
            break

        (time_ns, node, intf, qidx, qlen,
         sip, dip, size, l3Prot, event, ecn, nodeType) = header

        union_bytes = data[off + base_size : off + rec_size]

        row = {
            "time": time_ns / 1e9,
            "node": node,
            "intf": intf,
            "qidx": qidx,
            "qlen": qlen,
            "sip": sip,
            "dip": dip,
            "size": size,
            "l3Prot": l3Prot,
            "event": event,
            "ecn": ecn,
            "nodeType": nodeType,
            "sport": None,
            "dport": None,
            "seq": None,
            "ts": None,
            "pg": None,
            "payload": None,
            "ProtType": None,
        }

        # --- Decode union based on l3Prot ---
        try:
            if l3Prot in (0x6, 0x11):  # TCP/UDP data
                (sport, dport, seq, ts, pg, payload) = struct.unpack(fmt_data, union_bytes[:struct.calcsize(fmt_data)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "seq": seq,
                    "ts": ts,
                    "pg": pg,
                    "payload": payload,
                    "ProtType": "TCP" if l3Prot == 0x6 else "UDP",
                })
            elif l3Prot in (0xFC, 0xFD):  # ACK
                (sport, dport, flags, pg, seq, ts) = struct.unpack(fmt_ack, union_bytes[:struct.calcsize(fmt_ack)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "seq": seq,
                    "ts": ts,
                    "pg": pg,
                    "ProtType": "ACK",
                })
            elif l3Prot == 0xFE:  # PFC
                (pfc_time, pfc_qlen, qIndex) = struct.unpack(fmt_pfc, union_bytes[:struct.calcsize(fmt_pfc)])
                row.update({
                    "pfc_time": pfc_time,
                    "pfc_qlen": pfc_qlen,
                    "pfc_qIndex": qIndex,
                    "ProtType": "PFC",
                })
            elif l3Prot == 0xFF:  # CNP
                (fid, qIndex, ecnBits, qfb, total) = struct.unpack(fmt_cnp, union_bytes[:struct.calcsize(fmt_cnp)])
                row.update({
                    "cnp_fid": fid,
                    "cnp_qIndex": qIndex,
                    "cnp_ecnBits": ecnBits,
                    "cnp_qfb": qfb,
                    "cnp_total": total,
                    "ProtType": "CNP",
                })
            else:  # default qp
                (sport, dport) = struct.unpack(fmt_qp, union_bytes[:struct.calcsize(fmt_qp)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "ProtType": "QP",
                })
        except struct.error:
            # corrupted or incomplete record
            pass

        rows.append(row)

        if time_ns < last_time:
            # likely misaligned data — stop
            break
        last_time = time_ns
        off += rec_size

    return pd.DataFrame(rows)

def calculate_throughput(df, interval):
    """
    Calculates throughput in packets over a specified time interval.
    """
    if df.empty:
        return pd.Series()
        
    # Set time as index
    df = df.set_index(pd.to_datetime(df['time'], unit='s'))
    
    # Resample data into time bins and sum the packet coubnt
    throughput = df['size'].resample(f'{interval}s').count()
    
    throughput_gbps = (throughput) / (interval * 1e9)
    
    return throughput_gbps

In [2]:
import plotly.graph_objects as go
import pandas as pd

def plot(df, interval):
    # === Configuration ===
    group_by_node = True      # Toggle: True = per-node plots, False = per-flow plots
    
    # === Build mapping from sip -> node (node is the shorter sender id) ===
    sip_node_map = (
        df[['sip', 'node']]
        .drop_duplicates(subset='sip')
        .set_index('sip')['node']
        .to_dict()
    )

    # === Create short-name columns ===
    df['sip_short'] = df['sip'].map(sip_node_map).fillna(df['sip']).astype(str)
    df['dip_short'] = df['dip'].map(sip_node_map).fillna(df['dip']).astype(str)

    # === Event labels ===
    event_labels = {0: "Recv", 1: "Enqu", 2: "Dequ"}

    # === Helper: group column selection ===
    base_group_cols = ['sip_short', 'dip_short', 'ProtType']
    if group_by_node:
        group_cols = ['node'] + base_group_cols
        print("🔹 Grouping by node (per-node throughput plots).")
    else:
        group_cols = base_group_cols
        print("🔹 Grouping by flow only (aggregated throughput).")

    # === Loop over events ===
    for event_val, event_name in event_labels.items():
        df_event = df[df['event'] == event_val]
        if df_event.empty:
            print(f"No data for event {event_name} ({event_val}).")
            continue

        series_list = []
        count_dict = {}

        # Group by selected columns
        for keys, group_df in df_event.groupby(group_cols):
            if group_by_node:
                node, sip_s, dip_s, prot = keys
                key_name = f"{node}_{sip_s}_to_{dip_s}_prot{str(prot).replace('/', '_')}"
            else:
                sip_s, dip_s, prot = keys
                key_name = f"{sip_s}_to_{dip_s}_prot{str(prot).replace('/', '_')}"

            # Count raw entries
            count_dict[key_name] = len(group_df)

            # Compute throughput series
            s = calculate_throughput(group_df, interval)
            if s.empty:
                continue
            series_list.append(s.rename(key_name))

        # Print raw entry counts
        print(f"\n=== Entry counts for event '{event_name}' (raw dataset rows) ===")
        for k, v in count_dict.items():
            print(f"{k}: {v}")

        if not series_list:
            print(f"No throughput series to plot for event {event_name}.")
            continue

        df_all = pd.concat(series_list, axis=1).fillna(0)

        # === Plotly Figure ===
        fig = go.Figure()

        for col in df_all.columns:
            count = count_dict.get(col, 0)
            fig.add_trace(go.Scatter(
                x=df_all.index,
                y=df_all[col],
                mode='lines',
                name=f"{col} ({count} entries)"
            ))

        grouping_text = "Per-Node" if group_by_node else "Per-Flow"
        fig.update_layout(
            title=f"{grouping_text} Throughput Over Time ({event_name})",
            xaxis_title="Time",
            yaxis_title="Throughput",
            hovermode='x unified',
            legend_title="Flow (and Node, if applicable)",
            template='plotly_white',
            height=600,
            width=1000
        )

        # === Save and show ===
        base_name = globals().get('base', 'throughput')
        out_html = f"{base_name}_{event_name}.html"
        fig.write_html(out_html)
        print(f"\n✅ Interactive plot for event '{event_name}' saved to {out_html}")
        fig.show()


In [19]:
# trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251113_235331/ns3/astrasim_trace.tr'
# trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_003331/ns3/astrasim_trace.tr'
# trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_011331/ns3/astrasim_trace.tr'
# trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_015331/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_023331/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_024055/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_024753/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_032753/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_040753/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_041629/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_042431/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_050431/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_054431/ns3/astrasim_trace.tr'
# trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_102549/ns3/astrasim_trace.tr'
# trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_105830/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_113645/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251114_150125/ns3/astrasim_trace.tr'
df = parse_qbb_trace(trace_output_file)
interval = 0.1
plot(df, interval)



🔹 Grouping by node (per-node throughput plots).

=== Entry counts for event 'Recv' (raw dataset rows) ===
0_16_to_4294967295.0_protPFC: 3686
0_2_to_0.0_protACK: 28840
1_16_to_4294967295.0_protPFC: 3238
1_2_to_1.0_protACK: 23457
1_2_to_1.0_protUDP: 10000
2_0_to_2.0_protUDP: 49783
2_1_to_2.0_protACK: 9140
2_1_to_2.0_protUDP: 45337
16_0_to_2.0_protUDP: 59910
16_1_to_2.0_protACK: 10000
16_1_to_2.0_protUDP: 70978
16_2_to_0.0_protACK: 28840
16_2_to_1.0_protACK: 23457
16_2_to_1.0_protUDP: 10000
17_0_to_2.0_protUDP: 49783
17_1_to_2.0_protACK: 9140
17_1_to_2.0_protUDP: 45337
17_2_to_0.0_protACK: 28840
17_2_to_1.0_protACK: 23457
17_2_to_1.0_protUDP: 10000
18_0_to_2.0_protUDP: 49783
18_1_to_2.0_protACK: 9140
18_1_to_2.0_protUDP: 45337
18_2_to_1.0_protACK: 23457
18_2_to_1.0_protUDP: 10000
19_2_to_0.0_protACK: 28840

✅ Interactive plot for event 'Recv' saved to throughput_Recv.html



=== Entry counts for event 'Enqu' (raw dataset rows) ===
1_1_to_2.0_protACK: 30000
2_2_to_0.0_protACK: 86520
2_2_to_1.0_protACK: 70371
16_0_to_2.0_protUDP: 49783
16_1_to_2.0_protACK: 9140
16_1_to_2.0_protUDP: 45337
16_16_to_4294967295.0_protPFC: 6924
16_2_to_0.0_protACK: 28840
16_2_to_1.0_protACK: 23457
16_2_to_1.0_protUDP: 10000
17_0_to_2.0_protUDP: 49783
17_1_to_2.0_protACK: 9140
17_1_to_2.0_protUDP: 45337
17_2_to_0.0_protACK: 28840
17_2_to_1.0_protACK: 23457
17_2_to_1.0_protUDP: 10000
18_0_to_2.0_protUDP: 49783
18_1_to_2.0_protACK: 9140
18_1_to_2.0_protUDP: 45337
18_2_to_1.0_protACK: 23457
18_2_to_1.0_protUDP: 10000
19_2_to_0.0_protACK: 28840

✅ Interactive plot for event 'Enqu' saved to throughput_Enqu.html



=== Entry counts for event 'Dequ' (raw dataset rows) ===
0_0_to_2.0_protUDP: 179730
1_1_to_2.0_protACK: 30000
1_1_to_2.0_protUDP: 212934
2_2_to_0.0_protACK: 86520
2_2_to_1.0_protACK: 70371
2_2_to_1.0_protUDP: 30000
16_0_to_2.0_protUDP: 149349
16_1_to_2.0_protACK: 27420
16_1_to_2.0_protUDP: 136011
16_16_to_4294967295.0_protPFC: 20772
16_2_to_0.0_protACK: 86520
16_2_to_1.0_protACK: 70371
16_2_to_1.0_protUDP: 30000
17_0_to_2.0_protUDP: 149349
17_1_to_2.0_protACK: 27420
17_1_to_2.0_protUDP: 136011
17_2_to_0.0_protACK: 86520
17_2_to_1.0_protACK: 70371
17_2_to_1.0_protUDP: 30000
18_0_to_2.0_protUDP: 149349
18_1_to_2.0_protACK: 27420
18_1_to_2.0_protUDP: 136011
18_2_to_1.0_protACK: 70371
18_2_to_1.0_protUDP: 30000
19_2_to_0.0_protACK: 86520

✅ Interactive plot for event 'Dequ' saved to throughput_Dequ.html


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import re

def parse_qlen_log(file_path):
    """
    Parses the queue length log file into a pandas DataFrame.
    
    The expected format for each line is:
    'time <timestamp> <node_id> j <iface1> <qlen1> j <iface2> <qlen2> ...'
    """
    records = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts or parts[0] != 'time':
                    continue
                
                time_ns = int(parts[1])
                node_id = int(parts[2])
                
                # Iterate over the 'j <iface> <qlen>' groups
                for i in range(3, len(parts), 3):
                    if parts[i] == 'j' and i + 2 < len(parts):
                        iface_id = int(parts[i+1])
                        qlen = int(parts[i+2])
                        records.append({
                            'time_ns': time_ns,
                            'time_s': time_ns / 1e9,
                            'node': node_id,
                            'interface': iface_id,
                            'qlen_bytes': qlen
                        })
    except FileNotFoundError:
        print(f"Error: Queue length file not found at {file_path}")
        return pd.DataFrame()
    except (ValueError, IndexError) as e:
        print(f"Error parsing file {file_path}: {e}")
        return pd.DataFrame()

    if not records:
        print("No valid queue length records found.")
        return pd.DataFrame()
        
    return pd.DataFrame(records)

def plot_qlen(df_qlen):
    """
    Generates an interactive plot of queue length over time for each switch interface.
    """
    if df_qlen.empty:
        print("DataFrame is empty, cannot plot queue lengths.")
        return

    fig = go.Figure()

    # Group data by switch node and interface to plot each as a separate line
    for (node, interface), group in df_qlen.groupby(['node', 'interface']):
        fig.add_trace(go.Scatter(
            x=group['time_s'],
            y=group['qlen_bytes'],
            mode='lines',
            name=f'Switch {node}, Interface {interface}'
        ))

    fig.update_layout(
        title='Switch Queue Length Over Time',
        xaxis_title='Time (seconds)',
        yaxis_title='Queue Length (Bytes)',
        hovermode='x unified',
        legend_title="Switch Interface",
        template='plotly_white',
        height=600,
        width=1000
    )

    # Save and show the plot
    out_html = "queue_length_plot.html"
    fig.write_html(out_html)
    print(f"\n✅ Interactive queue length plot saved to {out_html}")
    fig.show()

# --- Example Usage ---
# 1. Define the path to your queue length log file
qlen_file_path = '/app/astra-sim/upc/configuration/ns3/output/astrasim_16nodes_ring_pfc_qlen.txt'

# 2. Parse the file into a DataFrame
df_qlen = parse_qlen_log(qlen_file_path)

# 3. Plot the data
if not df_qlen.empty:
    plot_qlen(df_qlen)
else:
    print("Could not generate queue length plot.")


✅ Interactive queue length plot saved to queue_length_plot.html


In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    Returns the full path to the first .txt file found, or None.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a simple 'key value' or 'key = value' configuration file into a dictionary.
    Ignores lines starting with '#' and handles potential whitespace.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                # Try splitting by '=' first, then by whitespace
                if '=' in line:
                    parts = line.split('=', 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params


def plot_recv_per_folder(
    run_folders: List[str], 
    interval: float, 
    group_by_node: bool = False, 
    nodes_to_plot: Optional[List[int]] = None
):
    """
    Generates a separate plot of 'Recv' throughput for each run folder,
    with lines broken down by node and flow, and allows filtering by node.
    """
    for folder in run_folders:
        print(f"Processing folder: {folder}")

        # 1. Parse config to create a descriptive title
        config_file = find_config_file(folder)
        if not config_file:
            print(f"  - Warning: Config file not found in {folder}")
            continue
            
        config_params = parse_config(config_file)
        
        title_parts = [
            f"cc:{config_params.get('cc_mode', 'N/A')}",
            f"win:{config_params.get('has_win', 'N/A')}",
            f"adapt:{config_params.get('var_win', 'N/A')}",
            f"buf:{config_params.get('buffer_size', 'N/A')}",
            f"size:{config_params.get('packet_payload_size', 'N/A')}"
        ]
        plot_title = ', '.join(title_parts)

        # 2. Parse trace file
        trace_file = os.path.join(folder, 'ns3', 'astrasim_trace.tr')
        if not os.path.exists(trace_file):
            print(f"  - Warning: Trace file not found: {trace_file}")
            continue
            
        df = parse_qbb_trace(trace_file)
        if df.empty:
            print(f"  - Warning: No data parsed from {trace_file}")
            continue
            
        df_recv = df[df['event'] == 0].copy()
        
        if df_recv.empty:
            print(f"  - Warning: No 'Recv' events found in {trace_file}")
            continue

        # Filter by selected nodes if a list is provided
        if nodes_to_plot:
            df_recv = df_recv[df_recv['node'].isin(nodes_to_plot)]
            if df_recv.empty:
                print(f"  - Warning: No 'Recv' events found for specified nodes {nodes_to_plot} in {trace_file}")
                continue

        # 3. Group data by flow and calculate throughput for each
        sip_node_map = (
            df[['sip', 'node']]
            .drop_duplicates(subset='sip')
            .set_index('sip')['node']
            .to_dict()
        )
        df_recv['sip_short'] = df_recv['sip'].map(sip_node_map).fillna(df_recv['sip']).astype(str)
        df_recv['dip_short'] = df_recv['dip'].map(sip_node_map).fillna(df_recv['dip']).astype(str)

        base_group_cols = ['sip_short', 'dip_short', 'ProtType']
        group_cols = ['node'] + base_group_cols if group_by_node else base_group_cols

        series_list = []
        for keys, group_df in df_recv.groupby(group_cols):
            if group_by_node:
                node, sip_s, dip_s, prot = keys
                key_name = f"Node {node}: {sip_s}→{dip_s} ({prot or 'N/A'})"
            else:
                sip_s, dip_s, prot = keys
                key_name = f"{sip_s}→{dip_s} ({prot or 'N/A'})"

            s = calculate_throughput(group_df, interval)
            if not s.empty:
                series_list.append(s.rename(key_name))

        if not series_list:
            print(f"  - Warning: No throughput data calculated for {trace_file}")
            continue
        
        df_all = pd.concat(series_list, axis=1).fillna(0)

        # 4. Create and show the plot for the current folder
        fig = go.Figure()
        for col in df_all.columns:
            fig.add_trace(go.Scatter(
                x=df_all.index,
                y=df_all[col],
                mode='lines',
                name=col
            ))
        
        grouping_text = "Per-Node" if group_by_node else "Per-Flow"
        fig.update_layout(
            title=f"Recv Throughput ({grouping_text})<br><sup>{plot_title}</sup>",
            xaxis_title="Time (seconds)",
            yaxis_title="Throughput (Gbps)",
            hovermode='x unified',
            legend_title="Flow",
            template='plotly_white',
            height=600,
            width=1000
        )
        
        folder_name_sanitized = os.path.basename(folder).replace('/', '_')
        # out_html = f"recv_throughput_per_flow_{folder_name_sanitized}.html"
        # fig.write_html(out_html)
        # print(f"  ✅ Interactive plot saved to {out_html}")
        fig.show()

# --- Usage ---
base_run_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy'
folders_to_plot = [os.path.join(base_run_folder, d) for d in os.listdir(base_run_folder) if os.path.isdir(os.path.join(base_run_folder, d))]
plot_interval = 1
nodes_to_include = list(range(16))

plot_recv_per_folder(
    folders_to_plot, 
    plot_interval, 
    group_by_node=True, 
    nodes_to_plot=nodes_to_include
)


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_011611
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_011611.html


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_011751
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_011751.html


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_011637
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_011637.html


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_014028
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_014028.html


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_011800
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_011800.html


...

Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_011743
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_011743.html


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_011832
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_011832.html


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_014003
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_014003.html


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_011815
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_011815.html


Processing folder: /app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251115_011735
  ✅ Interactive plot saved to recv_throughput_per_flow_run_20251115_011735.html


In [6]:
folders_to_plot

['run_20251115_011611',
 'run_20251115_011751',
 'run_20251115_011637',
 'run_20251115_014028',
 'run_20251115_011800',
 'run_20251115_011701',
 'run_20251115_011645',
 'run_20251115_014113',
 'run_20251115_014015',
 'run_20251115_012732',
 'run_20251115_011653',
 'run_20251115_014123',
 'run_20251115_014147',
 'run_20251115_013737',
 'run_20251115_014055',
 'run_20251115_011710',
 'run_20251115_012132',
 'run_20251115_012432',
 'run_20251115_011727',
 'run_20251115_014037',
 'run_20251115_014104',
 'run_20251115_011620',
 'run_20251115_014447',
 'run_20251115_014046',
 'run_20251115_011603',
 'run_20251115_013950',
 'run_20251115_013332',
 'run_20251115_011823',
 'run_20251115_013846',
 'run_20251115_011628',
 'run_20251115_011808',
 'run_20251115_011718',
 'run_20251115_013032',
 'run_20251115_013632',
 'run_20251115_011743',
 'run_20251115_011832',
 'run_20251115_014003',
 'run_20251115_011815',
 'run_20251115_011735']